# Q-MolGen: Phase 5 — Delaney ESOL Exploratory Data Analysis (EDA)

**Objective**: Perform statistical exploration of the Delaney Aqueous Solubility (ESOL) benchmark dataset ($N=1,128$ organic compounds).  
**Target Property**: `measured log solubility in mols per litre` (LogS).  
**Domain**: Computational Chemistry, Physicochemical Graph Analysis, Machine Learning Baseline Preparation.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

# Set modern visual styling
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    "figure.facecolor": "#0B0F19",
    "axes.facecolor": "#111827",
    "axes.edgecolor": "#374151",
    "axes.labelcolor": "#F9FAFB",
    "xtick.color": "#9CA3AF",
    "ytick.color": "#9CA3AF",
    "text.color": "#F9FAFB",
    "grid.color": "#1F2937",
})

## 1. Load and Inspect Raw Dataset

In [ ]:
DATA_PATH = os.path.join("..", "data", "raw", "delaney_esol.csv")
df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape[0]} molecules, {df.shape[1]} columns")
df.head()

## 2. Missing Value and Data Integrity Audit

In [ ]:
null_counts = df.isnull().sum()
print("Missing values per column:")
print(null_counts)
assert null_counts.sum() == 0, "Unexpected null values in benchmark dataset!"

## 3. Target Distribution: Aqueous Solubility (LogS)
- $\text{LogS} = \log_{10}(\text{Solubility in mol/L})$
- Most drug-like molecules lie between $-4.0$ and $-1.0$ LogS.

In [ ]:
target_col = "measured log solubility in mols per litre"
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df[target_col], kde=True, color="#06B6D4", bins=30, ax=ax)
ax.axvline(df[target_col].mean(), color="#F59E0B", linestyle="--", label=f"Mean: {df[target_col].mean():.2f}")
ax.axvline(df[target_col].median(), color="#10B981", linestyle=":", label=f"Median: {df[target_col].median():.2f}")
ax.set_title("Distribution of Measured Aqueous Solubility (LogS)", fontsize=14, fontweight="bold")
ax.set_xlabel("Log Solubility (mol/L)")
ax.set_ylabel("Count")
ax.legend(facecolor="#111827", edgecolor="#374151")
plt.show()

## 4. Molecular Weight (MW) & Lipinski Rule of 5 Boundary

In [ ]:
mw_col = "Molecular Weight"
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df[mw_col], kde=True, color="#8B5CF6", bins=30, ax=ax)
ax.axvline(500, color="#F43F5E", linestyle="--", label="Lipinski Ro5 Limit (500 Da)")
ax.set_title("Molecular Weight Distribution (Da)", fontsize=14, fontweight="bold")
ax.set_xlabel("Molecular Weight (Da)")
ax.legend(facecolor="#111827", edgecolor="#374151")
plt.show()

## 5. Topological Polar Surface Area (TPSA)

In [ ]:
tpsa_col = "Polar Surface Area"
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df[tpsa_col], kde=True, color="#10B981", bins=30, ax=ax)
ax.axvline(140, color="#F59E0B", linestyle="--", label="Oral Bioavailability Limit (140 Å²)")
ax.set_title("Polar Surface Area (TPSA) Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("TPSA (Å²)")
ax.legend(facecolor="#111827", edgecolor="#374151")
plt.show()

## 6. Pearson Correlation Heatmap

In [ ]:
num_cols = [target_col, mw_col, "Number of H-Bond Donors", "Number of Rings", "Number of Rotatable Bonds", tpsa_col]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="mako", ax=ax, square=True)
ax.set_title("Pearson Correlation Matrix of Physicochemical Descriptors", fontsize=13, fontweight="bold")
plt.show()

## 7. RDKit Molecular Structure Visualizations

In [ ]:
sample_smiles = df["smiles"].head(6).tolist()
sample_names = df["Compound ID"].head(6).tolist()
mols = [Chem.MolFromSmiles(s) for s in sample_smiles]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=sample_names)